# Part A — Exploratory Data Analysis

This notebook computes statistics for all four dataset files:
- `corpus.jsonl.gz` — the document corpus
- `train.jsonl.gz` — training queries with gold answers
- `valid.jsonl.gz` — validation queries with gold answers
- `bonus.jsonl.gz` — advanced queries (no gold answers)

Update `DATA_DIR` below to point to the folder containing the `.gz` files.

In [ ]:
import gzip
import json
import re
from collections import Counter
from pathlib import Path

# ── Config ──────────────────────────────────────────────────
DATA_DIR = Path(".")   # change to your data folder if needed

CORPUS_PATH = DATA_DIR / "corpus.jsonl.gz"
TRAIN_PATH  = DATA_DIR / "train.jsonl.gz"
VALID_PATH  = DATA_DIR / "valid.jsonl.gz"
BONUS_PATH  = DATA_DIR / "bonus.jsonl.gz"


def load_gz(path: Path) -> list[dict]:
    rows = []
    with gzip.open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


corpus = load_gz(CORPUS_PATH)
train  = load_gz(TRAIN_PATH)
valid  = load_gz(VALID_PATH)
bonus  = load_gz(BONUS_PATH)

print(f"Loaded — corpus:{len(corpus)}  train:{len(train)}  valid:{len(valid)}  bonus:{len(bonus)}")

## 1. Corpus

In [ ]:
url_docs  = [d for d in corpus if isinstance(d["id"], str)]
int_docs  = [d for d in corpus if isinstance(d["id"], int)]

print(f"Total documents : {len(corpus)}")
print(f"  URL-style IDs (Coveo docs) : {len(url_docs)}")
print(f"  Integer IDs   (Synthetic)  : {len(int_docs)}")

### 1.1 Text Length

In [ ]:
def length_stats(docs: list[dict], label: str) -> None:
    ls = [len(d["text"]) for d in docs]
    ls_sorted = sorted(ls)
    print(f"Text length [{label}]  (n={len(ls)})")
    print(f"  min    : {min(ls):>10,}")
    print(f"  max    : {max(ls):>10,}")
    print(f"  avg    : {sum(ls) // len(ls):>10,}")
    print(f"  median : {ls_sorted[len(ls) // 2]:>10,}")
    print()

length_stats(corpus,   "All")
length_stats(url_docs, "Coveo docs")
length_stats(int_docs, "Synthetic")

all_lengths = [len(d["text"]) for d in corpus]
print(f"Docs > 100k chars : {sum(1 for l in all_lengths if l > 100_000)}")
print(f"Docs >  10k chars : {sum(1 for l in all_lengths if l >  10_000)}")
print(f"Docs <   1k chars : {sum(1 for l in all_lengths if l <   1_000)}")

### 1.2 Metadata Coverage

In [ ]:
for field in ("product_prefix", "product_suffix", "product_version"):
    count = sum(1 for d in corpus if d.get(field))
    print(f"  {field:<20} {count:>4} / {len(corpus)}")

print()
prefixes = Counter(d["product_prefix"] for d in corpus if d.get("product_prefix"))
print("Top 10 product_prefix values:")
for k, v in prefixes.most_common(10):
    print(f"  {k:<15} {v}")

## 2. Query Files (Train / Valid / Bonus)

In [ ]:
def query_type(q: str) -> str:
    q = q.lower()
    if q.startswith("how many") or "count" in q:          return "aggregation"
    if "compare" in q or "difference" in q or " vs " in q: return "comparison"
    if q.startswith("list") or q.startswith("what are"):   return "enumeration"
    if "what is" in q or "what does" in q:                 return "definition"
    if q.startswith("when"):                               return "temporal"
    if "how to" in q or "how do" in q:                     return "procedural"
    return "other"


def query_file_stats(rows: list[dict], name: str, check_corpus: bool = True) -> None:
    print("=" * 52)
    print(f" {name}  ({len(rows)} rows)")
    print("=" * 52)

    q_lens = [len(r["query"]) for r in rows]
    a_lens = [len(r.get("answer", "")) for r in rows]

    print(f"Query length  — min:{min(q_lens):>4}  max:{max(q_lens):>4}  avg:{sum(q_lens)//len(q_lens):>4}")
    print(f"Answer length — min:{min(a_lens):>4}  max:{max(a_lens):>4}  avg:{sum(a_lens)//len(a_lens):>4}")

    doc_id_types = Counter(type(r["document_id"]).__name__ for r in rows)
    print(f"doc_id types  : {dict(doc_id_types)}")

    if check_corpus:
        corpus_ids = {str(d["id"]) for d in corpus}
        matched = sum(1 for r in rows if str(r["document_id"]) in corpus_ids)
        print(f"doc_ids found in corpus : {matched}/{len(rows)}")

    qtypes = Counter(query_type(r["query"]) for r in rows)
    print("Query types:")
    for k, v in sorted(qtypes.items(), key=lambda x: -x[1]):
        print(f"  {k:<15} {v}")
    print()

In [ ]:
query_file_stats(train, "TRAIN")
query_file_stats(valid, "VALID")
query_file_stats(bonus, "BONUS", check_corpus=False)

## 3. Bonus Queries

In [ ]:
print("Bonus queries (no gold answers):")
for i, r in enumerate(bonus, 1):
    print(f"  {i}. {r['query']}")

## 4. Sample Documents

In [ ]:
print("── Coveo doc sample ──────────────────────────────")
sample_coveo = next(d for d in corpus if isinstance(d["id"], str))
print(f"id   : {sample_coveo['id']}")
print(f"len  : {len(sample_coveo['text']):,} chars")
print(f"text : {sample_coveo['text'][:400]}...")

print()
print("── Synthetic product doc sample ──────────────────")
sample_synth = next(d for d in corpus if isinstance(d["id"], int))
print(f"id      : {sample_synth['id']}")
print(f"prefix  : {sample_synth['product_prefix']}")
print(f"suffix  : {sample_synth['product_suffix']}")
print(f"version : {sample_synth['product_version']}")
print(f"len     : {len(sample_synth['text']):,} chars")
print(f"text    : {sample_synth['text'][:400]}...")

## 5. Sample Train Q&A Pairs

In [ ]:
for r in train[:3]:
    print(f"Q  : {r['query']}")
    print(f"A  : {r['answer'][:200]}...")
    print(f"doc: {r['document_id']}")
    print()